In [36]:
!rm -rf InkubaLM-Challenge
!git clone https://github.com/melissafasol/InkubaLM-Challenge.git
%cd InkubaLM-Challenge

Cloning into 'InkubaLM-Challenge'...
remote: Enumerating objects: 328, done.
remote: Counting objects: 100% (126/126), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 328 (delta 79), reused 49 (delta 22), pack-reused 202 (from 1)
Receiving objects: 100% (328/328), 1.10 MiB | 13.29 MiB/s, done.
Resolving deltas: 100% (206/206), done.
/content/InkubaLM-Challenge/InkubaLM-Challenge


In [37]:
!git checkout refactor-src-structure
!git pull

Branch 'refactor-src-structure' set up to track remote branch 'refactor-src-structure' from 'origin'.
Switched to a new branch 'refactor-src-structure'
Already up to date.


In [38]:
!pip install datasets
!pip install transformers datasets peft trl accelerate bitsandbytes

In [39]:
# Cell 1: Install dependencies
!pip install -q transformers accelerate peft datasets bitsandbytes trl

# Cell 2: Imports and setup
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from datasets import load_dataset, Dataset
import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, Gemma3ForConditionalGeneration

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [40]:
import sys
sys.path.append("..")  # Add parent directory to the path

import os
from typing import List
from pathlib import Path
import numpy as np

# DO NOT EDIT
# create submission file
import pandas as pd
from huggingface_hub import login
from transformers import (
    AutoTokenizer,
)

from utils import (
    model_function,
    eval
    )

from src import (
    data_utils,
    model_utils,
    inference,
    prompts,
    evaluation,
    data_augment
    )

import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset, concatenate_datasets, Dataset, Value, DatasetDict

from trl import SFTConfig, SFTTrainer, DataCollatorForCompletionOnlyLM
from peft import PeftModel, PeftConfig
from sklearn.model_selection import train_test_split

In [45]:
#os.environ["TOKENIZERS_PARALLELISM"] = "false"

from huggingface_hub import login

try:
    from google.colab import userdata

    # Note: `userdata.get` is a Colab API. If you're not using Colab, set the env
    # vars as appropriate for your system.
    # userdata.get("HF_TOKEN") indicates that the name of the token in the Colab env is HF_TOKEN
    os.environ["hf_token_2"] = userdata.get("hf_token_2")
except:
    os.environ["hf_token_2"] = "----"

login(token=os.environ["hf_token_2"])

token = os.environ["hf_token_2"]
if token == "----":
    print("⚠️ Warning: No Hugging Face token found. Some models may not load.")
else:
    login(token=token)

In [56]:
hf_token_2 = '...' # paste your token here
os.environ["HF_TOKEN"] = hf_token_2


In [59]:
from huggingface_hub import login
login(token=hf_token_2)  # Force login using this token


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [60]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained("lelapa/InkubaLM-0.4B", token=hf_token_2)
model = AutoModelForCausalLM.from_pretrained("lelapa/InkubaLM-0.4B", token=hf_token_2)


config.json:   0%|          | 0.00/763 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.66G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [62]:
model_id = "lelapa/InkubaLM-0.4B"
model = AutoModelForCausalLM.from_pretrained(model_id,load_in_4bit=True,
    device_map="auto",trust_remote_code=True, token=hf_token_2)

vulavulaslm.py:   0%|          | 0.00/42.2k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/lelapa/InkubaLM-0.4B:
- vulavulaslm.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


In [64]:
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, lora_config)

You are using an old version of the checkpointing format that is deprecated (We will also silently ignore `gradient_checkpointing_kwargs` in case you passed it).Please update to the new format on your modeling file. To use the new format, you need to completely remove the definition of the method `_set_gradient_checkpointing` in your model.


In [65]:
import json
file_path = '/content/'

all_tasks = []
for fname in [os.path.join(file_path,"gemma_sentiment_soft_labels.jsonl"), os.path.join(file_path, "gemma_xnli_soft_labels.jsonl")]:
    with open(fname) as f:
        all_tasks.extend(json.loads(line) for line in f)

with open(os.path.join(file_path, "multitask_distill.jsonl"), 'w') as f:
    for row in all_tasks:
        f.write(json.dumps(row) + "\n")


In [71]:
formatted = []
for ex in all_tasks:
    formatted.append({
        "prompt": f"{ex['instruction']}\n{ex['input']}",
        "output": ex["output"],
        "task": ex["task"],
        "soft_label": ex.get("soft_label", None)
    })


In [72]:
formatted

[{'prompt': "Your task is to do sentiment analysis. Your output must be one word: positive, negative, or neutral. Don't output any explanation.\n@user ynxu fha da kanada kudi shikenan duk kayan nan zasu iya zama naka no🧢",
  'output': 'negative',
  'task': 'sentiment',
  'soft_label': {'positive': 0.0, 'neutral': 6e-05, 'negative': 0.99844}},
 {'prompt': "Your task is to do sentiment analysis. Your output must be one word: positive, negative, or neutral. Don't output any explanation.\n@user alhamdulillah babu abinda zamuce sai godiyan allah mai girma da daukaka allah yakara mana ni'imarsa🙏🙏",
  'output': 'positive',
  'task': 'sentiment',
  'soft_label': {'positive': 0.98594, 'neutral': 0.0, 'negative': 0.0}},
 {'prompt': "Your task is to do sentiment analysis. Your output must be one word: positive, negative, or neutral. Don't output any explanation.\n@user ke ina ruwan ki 😬 ba harkar film bane ba",
  'output': 'negative',
  'task': 'sentiment',
  'soft_label': {'positive': 0.00402,
 

In [66]:
from datasets import Dataset

formatted_dataset = Dataset.from_list([
    {
        "prompt": f"{ex['instruction']}\n{ex['input']}",
        "output": ex["output"],
        "task": ex["task"],
        "soft_label": ex.get("soft_label", None)
    }
    for ex in all_tasks
])


In [67]:
formatted_dataset

Dataset({
    features: ['prompt', 'output', 'task', 'soft_label'],
    num_rows: 600
})

In [69]:
formatted_dataset['task'].unique()

AttributeError: 'list' object has no attribute 'unique'